# 🧠 Artificial Neural Networks with TensorFlow/Keras
## Multi-Class Classification on MNIST Dataset

---

### What You Will Learn
- How a neural network processes data layer by layer
- What happens inside each layer (shapes, activations)
- Why we need activation functions (ReLU, Softmax)
- What overfitting is and how Dropout + L2 regularization fight it
- How to build, train, and evaluate a model in TensorFlow/Keras
- How to visualize training progress and analyze predictions

---

### The Dataset: MNIST
MNIST is a collection of **70,000 grayscale images** of handwritten digits (0–9), each 28×28 pixels:
- 60,000 training images
- 10,000 test images
- 10 classes (one per digit)

It is the "Hello World" of deep learning — well understood, well benchmarked, and easy to experiment with.

### Our Network Architecture
```
Input (28×28) → Flatten → Dense(128, ReLU) → Dropout(0.3)
              → Dense(64, ReLU) → Dropout(0.3)
              → Dense(10, Softmax) → Prediction
```

## Step 1 — Import Libraries

We need:
- **TensorFlow / Keras** — build and train the neural network
- **NumPy** — array math
- **Matplotlib** — visualization

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version:      {np.__version__}")

## Step 2 — Load the MNIST Dataset

Keras includes MNIST built-in. One line downloads and splits it for us.

The data comes as:
- `x_train` / `x_test` — images as NumPy arrays of shape `(N, 28, 28)`, values 0–255
- `y_train` / `y_test` — integer labels 0–9

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print("=== Dataset Shapes ===")
print(f"x_train: {x_train.shape}  (60000 images, each 28x28 pixels)")
print(f"y_train: {y_train.shape}  (60000 integer labels, values 0-9)")
print(f"x_test:  {x_test.shape}")
print(f"y_test:  {y_test.shape}")
print(f"\nPixel value range: min={x_train.min()}, max={x_train.max()}")
print(f"Label classes: {np.unique(y_train)}")

## Step 3 — Visualize the Data

Before building any model, always look at your data. This helps you:
- Understand what you're working with
- Spot any data quality issues
- Build intuition for what the network needs to learn

We also visualize the **pixel distribution** to understand why normalization matters.

In [ ]:
# Show 10 sample images with their labels
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i], cmap='gray')
    ax.set_title(f"Label: {y_train[i]}", fontsize=13, fontweight='bold')
    ax.axis('off')
plt.suptitle("MNIST Training Samples (raw, unnormalized)", fontsize=15)
plt.tight_layout()
plt.show()

# Count samples per class
fig, ax = plt.subplots(figsize=(10, 4))
unique, counts = np.unique(y_train, return_counts=True)
ax.bar(unique, counts, color='steelblue', alpha=0.8)
ax.set_xlabel('Digit Class (0-9)', fontsize=12)
ax.set_ylabel('Number of Training Samples', fontsize=12)
ax.set_title('Class Distribution in Training Set', fontsize=14)
ax.set_xticks(range(10))
for i, (u, c) in enumerate(zip(unique, counts)):
    ax.text(u, c + 50, str(c), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

# Look at one image in detail
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(x_train[0], cmap='gray')
axes[0].set_title(f'Image of digit: {y_train[0]}', fontsize=13)
axes[0].axis('off')
axes[1].hist(x_train[0].flatten(), bins=30, color='steelblue', alpha=0.8)
axes[1].set_title('Pixel value distribution (before normalization)', fontsize=12)
axes[1].set_xlabel('Pixel Value (0-255)')
axes[1].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

## Step 4 — Preprocessing

### Why Normalize?
Raw pixel values range from 0–255. Neural networks with large input values:
- Have unstable gradients during training
- Converge much slower
- May produce overflow in weight updates

By dividing by 255, all values become **0.0 to 1.0**, which:
- Keeps gradients in a stable range
- Makes the optimizer's job much easier
- Leads to faster and more reliable training

### Why NOT flatten here?
We let the `Flatten` layer inside the model handle reshaping. This keeps preprocessing separate from model architecture and makes the code cleaner.

In [ ]:
# Cast to float32 and normalize to [0.0, 1.0]
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

print("After normalization:")
print(f"  x_train: dtype={x_train.dtype}, min={x_train.min():.3f}, max={x_train.max():.3f}")
print(f"  x_test:  dtype={x_test.dtype},  min={x_test.min():.3f},  max={x_test.max():.3f}")
print(f"\nLabels (unchanged):")
print(f"  y_train sample: {y_train[:10]}")
print(f"  y_train dtype: {y_train.dtype}  (integers 0-9 for sparse_categorical_crossentropy)")

## Step 5 — Build the Model

### Architecture Design Choices

**`Flatten`** — Converts each 28×28 image into a 1D vector of 784 numbers. This is required because Dense layers only accept 1D input per sample.

**`Dense(128, activation='relu')`** — 128 neurons, each connected to all 784 inputs. That's 784×128 = 100,352 weights + 128 biases = ~100K parameters just in this layer. ReLU activation: `f(x) = max(0, x)` — fast, simple, and avoids the vanishing gradient problem.

**`Dropout(0.3)`** — During training, randomly zeroes 30% of neurons per batch. The network must learn redundant representations, improving generalization. **Automatically disabled during prediction.**

**`kernel_regularizer=l2(0.001)`** — Adds a small penalty proportional to the square of each weight. Prevents any single weight from growing too large and dominating the network's decisions.

**`Dense(10, activation='softmax')`** — Output layer. 10 neurons = 10 digit classes. Softmax converts raw scores to probabilities: `P(class_i) = e^(score_i) / Σ e^(score_j)`. The probabilities sum to exactly 1.0.

In [ ]:
# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

model = keras.Sequential([
    # ── Input processing ──────────────────────────────────────────
    # Flatten: (28, 28) → (784,)
    keras.layers.Flatten(input_shape=(28, 28), name='flatten'),

    # ── Hidden Layer 1 ────────────────────────────────────────────
    # 784 inputs × 128 neurons = 100,352 learnable weights
    keras.layers.Dense(
        128, activation='relu',
        kernel_regularizer=keras.regularizers.l2(0.001),
        name='dense_1'
    ),
    # Dropout: randomly zero 30% of neurons during training
    keras.layers.Dropout(0.3, name='dropout_1'),

    # ── Hidden Layer 2 ────────────────────────────────────────────
    keras.layers.Dense(
        64, activation='relu',
        kernel_regularizer=keras.regularizers.l2(0.001),
        name='dense_2'
    ),
    keras.layers.Dropout(0.3, name='dropout_2'),

    # ── Output Layer ──────────────────────────────────────────────
    # 10 neurons = 10 digit classes
    # Softmax: converts raw scores to probabilities summing to 1
    keras.layers.Dense(10, activation='softmax', name='output')
], name='MNIST_ANN')

model.summary()

## Step 6 — Compile the Model

### Optimizer: Adam
Adam (Adaptive Moment Estimation) is the most popular optimizer. Unlike basic gradient descent which uses one fixed learning rate for all weights, Adam:
- Maintains a separate learning rate for each weight
- Adapts the rate based on how noisy or stable each gradient is
- Usually converges faster and more reliably than SGD

### Loss: Sparse Categorical Crossentropy
Used when:
- You have **more than 2 classes** (multi-class, not binary)
- Your labels are **integers** (0, 1, 2, ... not one-hot vectors)

If your labels were one-hot encoded `[0,0,0,1,0,0,0,0,0,0]`, you'd use `categorical_crossentropy` instead.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully.")
print(f"  Optimizer: Adam (lr=0.001)")
print(f"  Loss:      sparse_categorical_crossentropy")
print(f"  Metric:    accuracy")

## Step 7 — Train the Model

### Key Parameters

**`epochs=15`** — One epoch = one full pass over the entire training set. We do 15 passes.

**`batch_size=128`** — Instead of processing one image at a time or all at once, we process mini-batches of 128 images. This is more computationally efficient and provides good gradient estimates.

**`validation_split=0.1`** — Hold back 10% of training data (6,000 images) as a validation set. We use this to monitor generalization during training and detect overfitting early. **The model never trains on validation data** — it's only used for evaluation.

### Early Stopping
We use `EarlyStopping` to automatically stop training if validation loss stops improving, and restore the best weights. This prevents overfitting automatically.

In [ ]:
# Callback: stop early if validation loss doesn't improve
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=4,          # wait 4 epochs without improvement before stopping
    restore_best_weights=True,  # revert to best weights at the end
    verbose=1
)

# Callback: reduce learning rate when plateauing
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,          # halve the learning rate
    patience=2,
    min_lr=1e-6,
    verbose=1
)

print("Starting training...\n")
history = model.fit(
    x_train, y_train,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

## Step 8 — Evaluate on Test Set

The test set is data the model has **never seen** during training or validation. This gives us the most honest measure of real-world performance.

Look for:
- **Test accuracy close to validation accuracy** → Good generalization, minimal overfitting
- **Large gap between train and test accuracy** → Overfitting, try more dropout or less capacity
- **Low test and train accuracy** → Underfitting, try more capacity or fewer regularization constraints

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
train_loss, train_acc = model.evaluate(x_train, y_train, verbose=0)

print("=" * 40)
print(f"  Train Accuracy : {train_acc:.4f}  ({train_acc*100:.2f}%)")
print(f"  Train Loss     : {train_loss:.4f}")
print("-" * 40)
print(f"  Test  Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  Test  Loss     : {test_loss:.4f}")
print("=" * 40)
gap = train_acc - test_acc
print(f"  Generalization gap: {gap:.4f} ({gap*100:.2f}%)")
if gap < 0.02:
    print("  → Excellent! Model generalizes well.")
elif gap < 0.05:
    print("  → Acceptable. Mild overfitting.")
else:
    print("  → Consider more dropout or stronger regularization.")

## Step 9 — Training Curves: Diagnosing Your Model

Training curves are the most important diagnostic tool. They show you exactly what happened during training.

### What to Look For
- **Both curves decreasing together** → Healthy training
- **Validation loss starts rising while train loss keeps falling** → Overfitting! Stop here.
- **Both curves still high** → Underfitting, need more capacity
- **Very noisy curves** → Learning rate may be too high
- **Both curves flat from epoch 1** → Learning rate too low, or data issue

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax = axes[0]
ax.plot(history.history['accuracy'],     'o-', label='Train accuracy',
        linewidth=2, markersize=5, color='steelblue')
ax.plot(history.history['val_accuracy'], 's-', label='Validation accuracy',
        linewidth=2, markersize=5, color='orange')
ax.set_title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.8, 1.02])

# Loss plot
ax = axes[1]
ax.plot(history.history['loss'],     'o-', label='Train loss',
        linewidth=2, markersize=5, color='steelblue')
ax.plot(history.history['val_loss'], 's-', label='Validation loss',
        linewidth=2, markersize=5, color='orange')
ax.set_title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.suptitle('Training History — Healthy curves show train & val converging together',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## Step 10 — Confusion Matrix

A confusion matrix shows which digits the model confuses with which other digits.
Row = **true** label, Column = **predicted** label.
- Diagonal = correct predictions (we want these high)
- Off-diagonal = mistakes (interesting to analyze — which digits look similar?)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Generate predictions on full test set
y_pred_probs = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=range(10), yticklabels=range(10))
ax.set_xlabel('Predicted Label', fontsize=13)
ax.set_ylabel('True Label', fontsize=13)
ax.set_title('Confusion Matrix — Test Set (10,000 images)', fontsize=14)
plt.tight_layout()
plt.show()

# Classification report
print("\nPer-Class Performance:")
print(classification_report(y_test, y_pred,
      target_names=[f'Digit {i}' for i in range(10)]))

## Step 11 — Visualize Predictions

Let's look at individual predictions and specifically find mistakes to understand where the model fails.

**Green title** = correct prediction  
**Red title** = wrong prediction (true label → predicted label)

In [ ]:
# Show 25 predictions from test set
fig, axes = plt.subplots(5, 5, figsize=(13, 13))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_test[i].reshape(28, 28), cmap='gray')
    true_label = y_test[i]
    pred_label = y_pred[i]
    confidence = y_pred_probs[i][pred_label]
    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(f"True: {true_label}  Pred: {pred_label}\n{confidence:.1%}",
                 color=color, fontsize=10)
    ax.axis('off')
plt.suptitle('Predictions on Test Set (green=correct, red=wrong)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Show some wrong predictions to analyze
wrong_indices = np.where(y_pred != y_test)[0]
print(f"\nTotal wrong predictions: {len(wrong_indices)} / {len(y_test)} "
      f"({len(wrong_indices)/len(y_test)*100:.1f}% error rate)")

if len(wrong_indices) > 0:
    fig, axes = plt.subplots(3, 5, figsize=(13, 8))
    for i, ax in enumerate(axes.flat):
        if i >= len(wrong_indices): break
        idx = wrong_indices[i]
        ax.imshow(x_test[idx].reshape(28, 28), cmap='gray')
        confidence = y_pred_probs[idx][y_pred[idx]]
        ax.set_title(f"True:{y_test[idx]} → Pred:{y_pred[idx]}\n{confidence:.1%}",
                     color='red', fontsize=10)
        ax.axis('off')
    plt.suptitle('Misclassified Examples — Analyzing Model Mistakes',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Bonus — Compare Model With vs Without Dropout

This is the most important experiment to understand regularization. We train two identical models:
1. **With Dropout** (our original model, already trained)
2. **Without Dropout** (same architecture, no regularization)

If both models achieve similar test accuracy, the gap between **train accuracy and test accuracy** tells the story. The model without Dropout will show a larger gap — that is overfitting in action.

In [ ]:
# Build a model WITHOUT dropout for comparison
tf.random.set_seed(42)

model_no_dropout = keras.Sequential([
    keras.layers.Flatten(input_shape=(28, 28)),
    keras.layers.Dense(128, activation='relu'),  # No L2, no Dropout
    keras.layers.Dense(64,  activation='relu'),
    keras.layers.Dense(10,  activation='softmax')
], name='No_Regularization')

model_no_dropout.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Training model WITHOUT dropout/regularization...")
history_no_drop = model_no_dropout.fit(
    x_train, y_train,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    verbose=0
)

# Compare results
_, test_acc_drop    = model.evaluate(x_test, y_test, verbose=0)
_, test_acc_nodrop  = model_no_dropout.evaluate(x_test, y_test, verbose=0)
_, train_acc_drop   = model.evaluate(x_train, y_train, verbose=0)
_, train_acc_nodrop = model_no_dropout.evaluate(x_train, y_train, verbose=0)

print("\n" + "=" * 50)
print(f"{'':25} {'With Reg':>10} {'No Reg':>10}")
print("-" * 50)
print(f"{'Train Accuracy':25} {train_acc_drop:>10.4f} {train_acc_nodrop:>10.4f}")
print(f"{'Test Accuracy':25} {test_acc_drop:>10.4f} {test_acc_nodrop:>10.4f}")
print(f"{'Overfitting gap':25} {(train_acc_drop-test_acc_drop):>10.4f} {(train_acc_nodrop-test_acc_nodrop):>10.4f}")
print("=" * 50)
print("A larger gap = more overfitting. Regularization reduces this gap.")

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(history.history['accuracy'],          label='With Dropout - Train',  linewidth=2)
ax.plot(history.history['val_accuracy'],      label='With Dropout - Val',    linewidth=2, linestyle='--')
ax.plot(history_no_drop.history['accuracy'],  label='No Dropout - Train',    linewidth=2)
ax.plot(history_no_drop.history['val_accuracy'], label='No Dropout - Val',   linewidth=2, linestyle='--')
ax.set_title('Accuracy: With vs Without Regularization', fontsize=13)
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(history.history['loss'],              label='With Dropout - Train',  linewidth=2)
ax.plot(history.history['val_loss'],          label='With Dropout - Val',    linewidth=2, linestyle='--')
ax.plot(history_no_drop.history['loss'],      label='No Dropout - Train',    linewidth=2)
ax.plot(history_no_drop.history['val_loss'],  label='No Dropout - Val',      linewidth=2, linestyle='--')
ax.set_title('Loss: With vs Without Regularization', fontsize=13)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

plt.suptitle('Overfitting Comparison — Notice the train/val gap without regularization',
             fontsize=12)
plt.tight_layout()
plt.show()

---

# 📝 Student Assignment

Now that you understand how the model works, it's your turn to experiment.

---

## Task 1: Fashion-MNIST Dataset

Repeat the entire pipeline using Fashion-MNIST instead of MNIST.
Fashion-MNIST has the same shape (28×28 grayscale) but contains clothing items:
T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot.

**Load it with:** `keras.datasets.fashion_mnist.load_data()`

**Questions to answer:**
- Is Fashion-MNIST easier or harder than MNIST? Why?
- Which clothing categories does the model confuse most often? Why might that be?

In [ ]:
# TASK 1: Use Fashion-MNIST
# YOUR CODE HERE

# Hint: replace the data loading line:
# (x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
#
# Class names for Fashion-MNIST:
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Then run the same preprocessing, model building, training, and evaluation
# Compare your test accuracy to what you got on MNIST

## Task 2: Architecture Experiments

Systematically test how architecture changes affect performance.

**Experiment A — Wider network:**
Change `Dense(128)` to `Dense(256)` and `Dense(64)` to `Dense(128)`. Does test accuracy improve? Does training time increase?

**Experiment B — Deeper network:**
Add a third hidden layer: `Dense(32, activation='relu')`. What happens?

**Experiment C — Stronger Dropout:**
Change `Dropout(0.3)` to `Dropout(0.5)`. What happens to the train/val accuracy gap?

**Experiment D — No activation functions:**
Remove `activation='relu'` (use default linear). The network becomes purely linear. What do you observe?

In [ ]:
# TASK 2: Architecture Experiments
# Build different model versions and compare their test accuracy

# Template:
def build_model(hidden_units=[128, 64], dropout_rate=0.3, activation='relu'):
    """
    Flexible model builder for experiments.
    hidden_units: list of neurons per layer, e.g. [256, 128, 64]
    dropout_rate: dropout probability (0 = no dropout)
    activation: 'relu', 'sigmoid', 'tanh', or None for linear
    """
    layers = [keras.layers.Flatten(input_shape=(28, 28))]
    for units in hidden_units:
        layers.append(keras.layers.Dense(units, activation=activation))
        if dropout_rate > 0:
            layers.append(keras.layers.Dropout(dropout_rate))
    layers.append(keras.layers.Dense(10, activation='softmax'))

    m = keras.Sequential(layers)
    m.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
    return m

# YOUR CODE HERE: build and compare different model variants
# Example:
# model_wide = build_model(hidden_units=[256, 128], dropout_rate=0.3)
# history_wide = model_wide.fit(x_train, y_train, epochs=10, batch_size=128,
#                                validation_split=0.1, verbose=0)
# Then compare test accuracy across variants

## Task 3: Deep Analysis of Predictions

Look beyond overall accuracy. Analyze **where** the model fails and **why**.

1. Find the 20 examples the model was **most wrong about** (highest confidence, wrong prediction)
2. Plot them and observe: are these genuinely ambiguous images, or clear errors?
3. Find all cases where the model confused **4 and 9**, or **3 and 8**. Do these pairs look similar?
4. Plot the model's **confidence distribution** — for correct predictions vs wrong predictions.

In [ ]:
# TASK 3: Deep analysis of predictions
# Run your predictions first if not already done
y_pred_probs = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
confidence = np.max(y_pred_probs, axis=1)

# YOUR CODE HERE:

# 1. Find most confident wrong predictions:
wrong_mask = (y_pred != y_test)
wrong_confidences = confidence[wrong_mask]
# Sort by confidence (highest first) and plot top 20

# 2. Find 4 vs 9 confusion:
# mask_4_pred_9 = (y_test == 4) & (y_pred == 9)
# Plot these examples

# 3. Plot confidence distribution for correct vs wrong:
# correct_conf = confidence[~wrong_mask]
# wrong_conf = confidence[wrong_mask]
# plt.hist([correct_conf, wrong_conf], ...)

## Task 4: Conceptual Questions

Answer these in your own words (no code needed).

---

### Q1: Why do we need activation functions?

*(Write your answer here)*

Hint: Without an activation function, every layer is just a linear transformation. A stack of linear transformations is still just one linear transformation — no matter how deep your network. The whole network collapses to a single linear layer.

---

### Q2: What happens without activation functions?

*(Write your answer here)*

Test it empirically using `build_model(activation=None)` from Task 2 and compare the test accuracy.

---

### Q3: Why does Dropout help generalization?

*(Write your answer here)*

---

### Q4: What is overfitting and how do you detect it from training curves?

*(Write your answer here)*

---

### Q5: Why do we use Softmax in the output layer instead of ReLU?

*(Write your answer here)*

---

### Q6: What would happen if we used `categorical_crossentropy` loss without one-hot encoding the labels?

*(Write your answer here)*

---

# Summary: What You Have Learned

| Concept | Key Insight |
|---------|-------------|
| ANN | Stacked layers of weighted connections + non-linear activations |
| Flatten | Converts 2D image (28×28) to 1D vector (784) |
| ReLU | Simple non-linearity in hidden layers; avoids vanishing gradient |
| Softmax | Multi-class output — 10 probabilities summing to 1.0 |
| Cross-entropy | Loss measuring distance between predicted and true distributions |
| Backprop | Assigns blame backwards through the network; computes gradients |
| Adam | Adaptive optimizer; separate learning rate per weight |
| Overfitting | Model memorizes training data; fails on new data |
| Dropout | Randomly silences neurons during training; forces redundancy |
| L2 Reg | Penalizes large weights; prevents dominant neurons |
| Validation | Held-out data used to monitor generalization; not for training |

---

**Next Steps:**
- Try Convolutional Neural Networks (CNNs) — they preserve spatial structure of images
- Explore Batch Normalization — another regularization technique
- Try different datasets: CIFAR-10, notMNIST, your own images
- Experiment with learning rate schedules
- Read about the Adam optimizer in the original paper by Kingma & Ba (2014)